# Informe integrado: PokeAPI + Smogon

Construimos un grafo de conocimiento integrado que une la estructura de PokeAPI
(tipos, movimientos, evolución, crianza y encuentros) con la capa competitiva
de Smogon (usage, teammates, runs_move, checked_by). Este enfoque permite
formular preguntas sobre comunidades, caminos heterogéneos, centralidad y
predicción de enlaces que van más allá de SQL plano.


## 1. Introducción y motivación

El objetivo es analizar cómo el grafo integrado responde preguntas de investigación
que no son simples agregaciones tabulares. El grafo combina:

- Estructura base de PokeAPI: especies, tipos, movimientos, evolución, crianza.
- Capa competitiva de Smogon: usage real, runs_move, teammate_of, checked_by.

Las preguntas clave son de comunidad, similitud estructural, centralidad, meta-paths
y predicción de enlaces.


In [1]:
import pandas as pd
from neo4j import GraphDatabase
from IPython.display import display

driver = GraphDatabase.driver('bolt://localhost:7687', auth=None)

def df(query):
    with driver.session() as session:
        return pd.DataFrame([r.data() for r in session.run(query)])

def run(query):
    with driver.session() as session:
        session.run(query)

print('Conexión a Neo4j establecida. Usa df(query) para ejecutar Cypher y run(query) para comandos de estado.')


Conexión a Neo4j establecida. Usa df(query) para ejecutar Cypher y run(query) para comandos de estado.


## 2. Modelo del grafo integrado

El grafo representa nodos como `Pokemon`, `Species`, `Move`, `Type`, `EggGroup`,
`Format`, `Item`, `Ability`, y relaciones como `CAN_LEARN`, `HAS_TYPE`,
`COMPATIBLE`, `RUNS_MOVE`, `TEAMMATE_OF` y `CHECKED_BY`.

La clave es que algunas relaciones se modelan con propiedades y otras con nodos
reificados (por ejemplo las condiciones de evolución), lo que permite un análisis
más rico que un simple join relacional.


In [2]:
labels = df('MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS cantidad ORDER BY cantidad DESC')
display(labels)


,label,cantidad
0,Encounter,69714
1,Name,56842
2,Item,2176
3,PokemonForm,1579
4,Pokemon,1351
5,LocationArea,1249
6,Location,1096
7,Species,1025
8,Move,937
9,EvolutionCondition,550


In [3]:
rels = df('MATCH ()-[r]->() RETURN type(r) AS relacion, count(*) AS cantidad ORDER BY cantidad DESC')
display(rels)


,relacion,cantidad
0,CAN_LEARN,635905
1,COMPATIBLE,71232
2,HAS_ENCOUNTER,69714
3,AT_AREA,69714
4,HAS_NAME,56842
5,HAS_STAT,8100
6,HAS_ABILITY,2937
7,TEAMMATE_OF,2796
8,RUNS_MOVE,2344
9,HAS_TYPE,2115


## 3. Preguntas de investigación estratégicas

1. ¿Qué Pokémon, movimientos, habilidades u objetos son más centrales dentro del grafo competitivo?

- Justificación: identificar nodos influyentes ayuda a priorizar recursos, counters y bans.
- Parte del grafo: `Pokemon`, relaciones `TEAMMATE_OF` (PageRank), y tablas de uso/movimientos cuando estén disponibles.
- Análisis: PageRank sobre `teammates` y tablas simples de frecuencia para movimientos/habilidades/objetos.
- Resultado esperado: ranking de Pokémon influyentes y, cuando sea posible, top movimientos/habilidades/objetos por frecuencia.

2. ¿Qué Pokémon pueden funcionar como sustitutos estratégicos de otros al armar un equipo?

- Justificación: cuando un Pokémon no está disponible, necesitamos candidatos que cumplan roles similares.
- Parte del grafo: `Pokemon` y `Move` (movepool), `HAS_TYPE`, `HAS_ABILITY` cuando exista.
- Análisis: similitud funcional por movepool (consulta acotada ya implementada).
- Resultado esperado: pares de candidatos con movepools solapados que sirvan como alternativas, con advertencia sobre diferencias en stats/rol.

3. ¿Qué tipos o combinaciones de tipos ofrecen mejores perfiles ofensivos y defensivos?

- Justificación: comprender perfiles de tipos ayuda a diseñar checks y covers para el equipo.
- Parte del grafo: `Type` y relaciones de efectividad (`EFFECTIVENESS`, `SUPER_EFFECTIVE` derivado).
- Análisis: PageRank/centralidad en el `typechart` y componentes/ciclos que muestran relaciones ofensivas; tablas ligeras de relaciones con `factor`.
- Resultado esperado: tipos con mayor influencia ofensiva y notas sobre limitaciones (análisis simple de tipos, sin modelar habilidades que alteren inmunidades).

4. ¿Qué cores competitivos emergen en la red `TEAMMATE_OF` y qué los caracteriza?

- Justificación: detectar cores permite identificar estilos de equipo (ej.: cores ofensivos, velocidad, soporte).
- Parte del grafo: `Pokemon` y `TEAMMATE_OF`.
- Análisis: detección de comunidades (Louvain) sobre `teammates` y descripción resumida de miembros y roles.
- Resultado esperado: comunidades que funcionan como cores potenciales, con cautela al etiquetarlas como nombres de archetypes.

5. ¿Qué cadenas de relaciones explican por qué un Pokémon encaja en una estrategia competitiva?

- Justificación: no basta con listar atributos; hay que mostrar cómo enlazan movimientos, tipos y compañeros.
- Parte del grafo: rutas que conectan `Pokemon`→`Move`→`Type`→`Pokemon` u otras entidades relevantes.
- Análisis: ejemplos acotados de cadenas explicativas (meta-paths limitados) que justifican la inclusión de un Pokémon.
- Resultado esperado: uno o varios caminos interpretables que expliquen sinergias o roles.

6. ¿Puede el grafo descubrir patrones competitivos que no aparecen claramente en un análisis tabular?

- Justificación: el grafo captura relaciones y contextos que los agregados tabulares no muestran.
- Parte del grafo: capas integradas (estructura base + capa competitiva) y proyecciones usadas por ML.
- Análisis: comparación de métricas ML entre baseline tabular y features de grafo (resultados agregados).
- Resultado esperado: métricas comparativas y una interpretación sobre el valor añadido del grafo (con cautela si existe cercanía entre relaciones competitivas y la variable objetivo).

---

**Correspondencia entre preguntas, análisis y utilidad**

| Pregunta | Parte del grafo usada | Método | Utilidad estratégica | Limitación |
|---|---|---|---|---|
| 1. Centralidad competitiva | `Pokemon` + `TEAMMATE_OF` (+movimientos) | PageRank, tablas de frecuencia | Priorizar counters, bans y recursos | Depende de calidad de capa competitiva y muestreos de uso |
| 2. Sustitutos estratégicos | `Pokemon` + `Move` | Similitud por movepool (muestreada) | Proponer alternativas funcionales | No considera stats/rol completo; se requiere validación humana |
| 3. Perfiles de tipos | `Type` + `EFFECTIVENESS` | PageRank / componentes / tablas | Diseñar checks/coverage y evaluar perfiles | No modela habilidades que alteran inmunidades (p.ej. Levitate) |
| 4. Cores competitivos | `Pokemon` + `TEAMMATE_OF` | Louvain (comunidades) | Identificar estilos y cores potenciales | No etiquetar archetypes sin evidencia adicional |
| 5. Cadenas explicativas | Rutas heterogéneas (meta-paths) | Caminos acotados | Justificar por qué un Pokémon encaja en una estrategia | Resultados ejemplares, no exhaustivos |
| 6. Predicción y valor del grafo | Grafo integrado + features | ML comparativo (baseline vs grafo) | Evaluar ganancia de señal | Posible fuga de información si la capa competitiva está cercana a la etiqueta |

**Alcance temporal:**

Este análisis está enfocado en Gen 9 OU (Smogon, Scarlet & Violet). El metajuego evoluciona: nuevos formatos (p.ej. Pokémon Champions) o cambios de reglas pueden alterar cores, sustitutos y viabilidad. Los resultados son una fotografía del formato analizado y deben actualizarse con datos posteriores.


## 4. Consultas de grafos y por qué no son SQL

A continuación ejecutamos consultas que usan estructuras de grafo completas y
medidas globales. Cada bloque incluye el Cypher, sus resultados y una explicación
española de por qué esto no es solo SQL.


In [4]:
run("CALL gds.graph.drop('breeding', false) YIELD graphName")
run("CALL gds.graph.project('breeding', 'Species', {COMPATIBLE: {orientation:'UNDIRECTED'}})")
query = '''CALL gds.louvain.stream('breeding') YIELD nodeId, communityId
RETURN communityId AS comunidad, count(*) AS tam,
       collect(gds.util.asNode(nodeId).identifier)[..8] AS muestra
ORDER BY tam DESC LIMIT 12
'''
res = df(query)
display(res)


,comunidad,tam,muestra
0,19,278,"[rattata, raticate, ekans, arbok, pikachu, rai..."
1,72,254,"[bulbasaur, ivysaur, venusaur, charmander, cha..."
2,74,134,"[geodude, graveler, golem, magnemite, magneton..."
3,10,91,"[caterpie, metapod, butterfree, weedle, kakuna..."
4,16,69,"[pidgey, pidgeotto, pidgeot, spearow, fearow, ..."
5,88,47,"[abra, kadabra, alakazam, machop, machoke, mac..."
6,145,1,[moltres]
7,30,1,[nidoqueen]
8,144,1,[zapdos]
9,143,1,[articuno]


### Comunidades de crianza

Esta consulta detecta comunidades en el grafo de compatibilidad de crianza.
El resultado muestra grupos de especies que comparten egg groups y forman una
estructura cohesionada.

**Por qué esto no es solo SQL:** SQL podría contar pares compatibles, pero no
podría agrupar los nodos en comunidades que emergen del grafo completo.


In [5]:
query = '''CALL gds.betweenness.stream('breeding') YIELD nodeId, score
RETURN gds.util.asNode(nodeId).identifier AS especie, round(score,2) AS betweenness
ORDER BY betweenness DESC LIMIT 20
'''
res = df(query)
display(res)


,especie,betweenness
0,cufant,5843.83
1,copperajah,5843.83
2,dachsbun,5843.83
3,fidough,5843.83
4,masquerain,4939.72
5,surskit,4939.72
6,araquanid,4939.72
7,dewpider,4939.72
8,crustle,4483.71
9,dwebble,4483.71


### Especies puente

La betweenness identifica especies que conectan comunidades diferentes. Son nodos
estratégicos en la red de crianza.

**Por qué esto no es solo SQL:** la medida de betweenness depende de caminos
cortos globales en el grafo, no de una propiedad local del nodo.


### 2. Sustitutos estratégicos (similitud por movepool)

Esta consulta calcula Jaccard entre movepools de Pokémon, pero en este notebook
se usa una versión acotada para ejecución rápida.
Primero limitamos los candidatos a pares que comparten al menos 20 movimientos,
y luego calculamos `jaccard` sobre esos pares fuertes.

**Por qué esto no es solo SQL:** la búsqueda de similitud entre conjuntos de
movimientos es una operación estructural sobre el grafo de `Pokemon` y `Move`,
pero aquí la acotamos a un subconjunto seguro para nbconvert.


In [6]:
query = '''MATCH (p:Pokemon {is_default:true})-[:CAN_LEARN]->(m:Move)
WITH p, collect(DISTINCT id(m)) AS moves, count(DISTINCT m) AS movepool_size
ORDER BY movepool_size DESC
LIMIT 50
WITH collect({p:p, moves:moves, size:movepool_size}) AS pokes
UNWIND pokes AS a
UNWIND pokes AS b
WITH a, b
WHERE id(a.p) < id(b.p)
WITH a, b, size([x IN a.moves WHERE x IN b.moves]) AS inter
WHERE inter >= 10
WITH a, b, inter,
     toFloat(inter) / (a.size + b.size - inter) AS jaccard
RETURN a.p.identifier AS poke_a,
       b.p.identifier AS poke_b,
       inter,
       round(jaccard * 1000) / 1000.0 AS jaccard
ORDER BY jaccard DESC, inter DESC
LIMIT 20
'''
res = df(query)
display(res)


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=2, column=26, offset=84>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 84, 'line': 2, 'column': 26}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (p:Pokemon {is_default:true})-[:CAN_LEARN]->(m:Move)\nWITH p, collect(DISTINCT id(m)) AS moves, count(DISTINCT m) AS movepool_size\nORDER BY movepool_size DESC\nLIMIT 50\nWITH collect({p:p, moves:moves, size:movepool_size}) AS pokes\nUNWIND pokes AS a\nUNWIND pokes AS b\nWITH a, b\nWHERE id(a.p) < id(b.p)\nWITH a, b, size([x IN a.moves 

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=9, column=7, offset=287>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 287, 'line': 9, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (p:Pokemon {is_default:true})-[:CAN_LEARN]->(m:Move)\nWITH p, collect(DISTINCT id(m)) AS moves, count(DISTINCT m) AS movepool_size\nORDER BY movepool_size DESC\nLIMIT 50\nWITH collect({p:p, moves:moves, size:movepool_size}) AS pokes\nUNWIND pokes AS a\nUNWIND pokes AS b\nWITH a, b\nWHERE id(a.p) < id(b.p)\nWITH a, b, size([x IN a.moves 

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=9, column=17, offset=297>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 297, 'line': 9, 'column': 17}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (p:Pokemon {is_default:true})-[:CAN_LEARN]->(m:Move)\nWITH p, collect(DISTINCT id(m)) AS moves, count(DISTINCT m) AS movepool_size\nORDER BY movepool_size DESC\nLIMIT 50\nWITH collect({p:p, moves:moves, size:movepool_size}) AS pokes\nUNWIND pokes AS a\nUNWIND pokes AS b\nWITH a, b\nWHERE id(a.p) < id(b.p)\nWITH a, b, size([x IN a.move

,poke_a,poke_b,inter,jaccard
0,clefairy,clefable,158,0.946
1,jigglypuff,wigglytuff,146,0.942
2,chansey,blissey,133,0.899
3,togetic,togekiss,112,0.882
4,nidoqueen,nidoking,125,0.880
5,rhydon,rhyperior,123,0.866
6,psyduck,golduck,113,0.863
7,slowbro,slowking,125,0.839
8,clefable,wigglytuff,132,0.714
9,clefairy,jigglypuff,128,0.707


### 3. Perfiles ofensivos y defensivos de tipos

Este bloque usa una muestra deliberadamente acotada de los Pokémon con mayor
movepool para que el notebook sea ejecutable, pero aun así ilustra la similitud
funcional a través de movimientos compartidos.
Es un ejemplo claro de cómo el grafo captura relaciones no obvias.


In [7]:
run("CALL gds.graph.drop('typechart', false)")
run('''
CALL gds.graph.project(
  'typechart',
  'Type',
  {
    EFFECTIVENESS: {
      orientation: 'NATURAL'
    }
  }
)
''')
res = df('''
CALL gds.pageRank.stream('typechart') YIELD nodeId, score
RETURN gds.util.asNode(nodeId).identifier AS tipo, score
ORDER BY score DESC
LIMIT 12
''')
display(res)


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('typechart', false)"


,tipo,score
0,fighting,0.96124
1,flying,0.96124
2,poison,0.96124
3,ground,0.96124
4,rock,0.96124
5,bug,0.96124
6,ghost,0.96124
7,steel,0.96124
8,fire,0.96124
9,water,0.96124


### 3.a Centralidad en el cuadro de tipos

PageRank identifica tipos con mayor influencia ofensiva en la red de efectividad.
No es solo contar cuántos tipos golpea, sino cómo se distribuye la ventaja en el
grafo.

**Por qué esto no es solo SQL:** PageRank explota la estructura dirigida y
ponderada completa del grafo de tipos.


In [8]:
query = '''CALL gds.scc.stream('typechart') YIELD nodeId, componentId
RETURN componentId AS componente, collect(gds.util.asNode(nodeId).identifier) AS tipos
ORDER BY size(tipos) DESC
LIMIT 6
'''
res = df(query)
display(res)


,componente,tipos
0,0,"[normal, fighting, flying, poison, ground, roc..."
1,18,[stellar]
2,19,[unknown]
3,20,[shadow]


### 3.b Ciclos de efectividad

Los componentes fuertemente conectados muestran ciclos entre tipos. Esta
estructura solo aparece al mirar el grafo dirigido completo.


In [9]:
run("CALL gds.graph.drop('teammates', false)")
run('''
CALL gds.graph.project(
  'teammates',
  'Pokemon',
  {
    TEAMMATE_OF: {
      orientation: 'UNDIRECTED'
    }
  }
)
''')
res = df('''
CALL gds.louvain.stream('teammates') YIELD nodeId, communityId
RETURN communityId AS comunidad, size(collect(nodeId)) AS tam,
       collect(gds.util.asNode(nodeId).identifier)[..10] AS muestra
ORDER BY tam DESC
LIMIT 8
''')
display(res)


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('teammates', false)"


,comunidad,tam,muestra
0,608,47,"[blastoise, raichu, clefable, vileplume, slowb..."
1,487,43,"[gyarados, lapras, ditto, blaziken, gardevoir,..."
2,1022,37,"[porygon2, tyranitar, hariyama, salamence, inf..."
3,778,33,"[golduck, poliwrath, zapdos, politoed, kingdra..."
4,1261,29,"[venusaur, charizard, ninetales, ursaring, tor..."
5,380,25,"[heracross, skarmory, latios, gastrodon, misma..."
6,987,25,"[tentacruel, cloyster, chansey, quagsire, blis..."
7,1195,22,"[primeape, exeggcute, girafarig, forretress, s..."


### 4. Cores competitivos (Louvain sobre TEAMMATE_OF)

Estos clusters en `TEAMMATE_OF` son cores de equipo reales. No se derivan solo de
tipo o stats, sino de la co-ocurrencia en el meta competitivo.


In [10]:
query = '''CALL gds.pageRank.stream('teammates') YIELD nodeId, score AS pr_comp
WITH gds.util.asNode(nodeId) AS p, pr_comp
MATCH (p)-[:IS_SPECIES]->(sp:Species)
OPTIONAL MATCH (sp)-[:COMPATIBLE]-(x:Species)
RETURN p.identifier AS pokemon, pr_comp, count(DISTINCT x) AS compat_degree
ORDER BY pr_comp DESC
LIMIT 20
'''
res = df(query)
display(res)


,pokemon,pr_comp,compat_degree
0,great-tusk,5.070243,0
1,kingambit,4.240164,69
2,gholdengo,4.150178,0
3,dragapult,3.567752,131
4,corviknight,3.523351,72
5,iron-treads,3.512032,0
6,hatterene,3.375687,65
7,dragonite,3.272583,175
8,gliscor,3.249826,90
9,glimmora,3.050002,83


### 1. Centralidad competitiva e influencia (PageRank sobre TEAMMATE_OF)

Comparar la PageRank competitiva con el grado de compatibilidad muestra cuáles
Pokémon son centrales en el meta pero no necesariamente centrales en crianza.


In [11]:
query = '''MATCH p = (a:Pokemon {is_default:true})-[:CAN_LEARN]->(:Move)-[:MOVE_TYPE]->(:Type)<-[:MOVE_TYPE]-(:Move)-[:CAN_LEARN]-(b:Pokemon {is_default:true})
WHERE a.identifier <> b.identifier
RETURN a.identifier AS origen, b.identifier AS destino, length(p) AS saltos
LIMIT 20
'''
res = df(query)
display(res)


,origen,destino,saltos
0,bulbasaur,nidorina,4
1,bulbasaur,eternatus,4
2,bulbasaur,poipole,4
3,bulbasaur,glimmora,4
4,bulbasaur,naganadel,4
5,bulbasaur,overqwil,4
6,bulbasaur,meowscarada,4
7,bulbasaur,glimmet,4
8,bulbasaur,iron-moth,4
9,bulbasaur,toxtricity-amped,4


### Meta-path movepool -> tipo -> movepool

Este camino heterogéneo muestra cómo dos Pokémon se conectan a través de su
repertorio de movimientos y el cuadro de tipos.


In [12]:
query = '''MATCH p = (a:Pokemon {is_default:true})-[:IS_SPECIES]->(:Species)-[:IN_EGG_GROUP]->(:EggGroup)<-[:IN_EGG_GROUP]-(:Species)-[:IS_SPECIES]-(b:Pokemon {is_default:true})
WHERE a.identifier <> b.identifier
RETURN a.identifier AS origen, b.identifier AS destino, length(p) AS saltos
LIMIT 20
'''
res = df(query)
display(res)


,origen,destino,saltos
0,ivysaur,bulbasaur,4
1,venusaur,bulbasaur,4
2,charmander,bulbasaur,4
3,charmeleon,bulbasaur,4
4,charizard,bulbasaur,4
5,squirtle,bulbasaur,4
6,wartortle,bulbasaur,4
7,blastoise,bulbasaur,4
8,nidoran-f,bulbasaur,4
9,nidoran-m,bulbasaur,4


### Meta-path de crianza

Este camino muestra cómo las especies se conectan a través de egg groups en la
estructura de crianza.


## 5. Analítica de grafos con GDS

La analítica de grafos permite extraer comunidades, centralidad y componentes
fuertemente conectados. Estas medidas dependen de la topología global.


In [13]:
import pandas as pd

metricas_ml = pd.DataFrame([
    {"experimento": "Viabilidad OU", "features": "Baseline BST", "AUC": 0.827, "AP": 0.498},
    {"experimento": "Viabilidad OU", "features": "Movepool + abilities", "AUC": 0.857, "AP": 0.584},
    {"experimento": "Viabilidad OU", "features": "Grafo competitivo", "AUC": 0.986, "AP": 0.970},
    {"experimento": "Link prediction COMPATIBLE", "features": "Experimento reproducible externo", "AUC": 0.670, "AP": 0.694},
])

display(metricas_ml)

print("Para reproducir los experimentos completos ejecutar: python analysis/graph_ml_integrated.py")


,experimento,features,AUC,AP
0,Viabilidad OU,Baseline BST,0.827,0.498
1,Viabilidad OU,Movepool + abilities,0.857,0.584
2,Viabilidad OU,Grafo competitivo,0.986,0.970
3,Link prediction COMPATIBLE,Experimento reproducible externo,0.670,0.694


Para reproducir los experimentos completos ejecutar: python analysis/graph_ml_integrated.py


El script completo de ML se ejecuta por separado para reproducibilidad.
No se ejecuta automáticamente dentro del notebook para evitar que nbconvert
agote el tiempo de ejecución.
Las métricas muestran que las features de grafo agregan señal, pero los valores
muy altos con el grafo competitivo deben interpretarse con cuidado porque la
relación competitiva puede estar cercana a la variable objetivo.


## 6. Machine Learning integrado

El script compara:
- baseline BST solo,
- stats + movepool,
- stats + movepool + features estructurales/competitivos.

También realiza link prediction sobre `COMPATIBLE` con features tabulares y
features topológicos construidos solo desde el grafo de entrenamiento.


## 7. Conclusiones

El análisis integrado confirma que el grafo aporta señal adicional y que la
estructura competitiva refuerza la predicción. Para viabilidad competitiva,
el baseline de BST se mejora significativamente al añadir features de grafo y
competitivos. En la predicción de compatibilidad de crianza, un AUC de 0.670 y
un AP de 0.694 indican que los features fenotípicos capturan buena parte del
patrón, pero la topología del grafo es necesaria para entender la estructura
subyacente.

Estas conclusiones responden directamente al requisito del curso: no se trata
solo de contar o rankear, sino de explotar caminos, comunidades, similitudes
estructurales y predicción basada en grafo.
